In [2]:
from pathlib import Path
import requests
import geopandas as gpd

project_dir = Path(r"C:\Users\Fiorella\OneDrive\VSCODE PROJECTS\Projects\DTSC 2\Personal Project Part 1")
output_dir = project_dir / "data" / "raw_data" / "tree_canopy"
file_path = output_dir / "charlotte_canopy.geojson"
output_dir.mkdir(parents=True, exist_ok=True)

# Query the Charlotte Tree Canopy ArcGIS REST API.
url = "https://services.arcgis.com/9Nl857LBlQVyzq54/ArcGIS/rest/services/Charlotte_Mecklenburg_Tree_Canopy/FeatureServer/0/query"
params = {
    "where": "1=1",
    "outFields": "*",
    "outSR": "4326",
    "f": "geojson"
}

response = requests.get(url, params=params, timeout=(10, 120))
response.raise_for_status()

# Save and reload the raw GeoJSON.
file_path.write_text(response.text, encoding="utf-8")
gdf_canopy = gpd.read_file(file_path, engine="pyogrio")

print(f"Saved {len(gdf_canopy):,} canopy features to {file_path}")
gdf_canopy.head()

Saved 1,135 canopy features to C:\Users\Fiorella\OneDrive\VSCODE PROJECTS\Projects\DTSC 2\Personal Project Part 1\data\raw_data\tree_canopy\charlotte_canopy.geojson


,OBJECTID,TotalAreaAcres,LandAcres,VegetationAreaAcres,VegetationAreaPercent,SoilAreaAcres,SoilAreaPercent,ImperviousAreaAcres,ImperviousAreaPercent,TotalImperviousAreaAcres,...,UrbanTreeCanopyAreaAcres,UrbanTreeCanopyAreaPercent,TreeCanopyArea2022Percent,TreeCanopyArea2022Acres,Geography,Geography_Value,Vintage,Shape__Area,Shape__Length,geometry
0,1,200258.84,199021.80,41236.14,20.591420,5878.24,2.94,57746.07,28.84,57746.07,...,94161.36,47.31,43.255282,86087.44,City of Charlotte,City of Charlotte,2022,8.727954e+09,1.402811e+06,"MULTIPOLYGON (((-80.95298 35.16321, -80.95249 ..."
1,2,40084.03,39790.98,8428.69,21.027544,1344.85,3.36,5128.19,12.79,5128.19,...,24889.24,62.55,61.404846,24433.59,City of Charlotte,Charlotte ETJ,2022,1.747000e+09,1.676054e+06,"MULTIPOLYGON (((-80.98334 35.30362, -80.98333 ..."
2,3,27854.85,27693.47,6429.19,23.081043,1022.48,3.67,7626.56,27.38,7626.56,...,12615.25,45.55,42.630000,11874.76,Council Districts,4,2022,1.214001e+09,4.485782e+05,"MULTIPOLYGON (((-80.7547 35.3939, -80.7543 35...."
3,4,31118.19,30919.25,6975.36,22.415691,771.65,2.48,8844.70,28.42,8844.70,...,14327.54,46.34,43.240000,13455.68,Council Districts,2,2022,1.356239e+09,4.683204e+05,"MULTIPOLYGON (((-80.93791 35.34725, -80.93761 ..."
4,5,24403.11,24273.60,4009.32,16.429534,285.67,1.17,6636.61,27.20,6636.61,...,13342.00,54.97,46.900000,11443.89,Council Districts,6,2022,1.063571e+09,1.901049e+05,"POLYGON ((-80.8206 35.20665, -80.81976 35.2057..."


In [3]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

# Load API key from .env file
load_dotenv()
api_key = os.getenv("CENSUS_API_KEY")

# Census API request (2022 ACS 5-Year)
url = "https://api.census.gov/data/2022/acs/acs5"
params = {
    "get": "NAME,B01003_001E,B19013_001E",
    "for": "tract:*",
    "in": "state:37 county:119"  # Mecklenburg County, NC
}

# Add key to params if present
if api_key:
    params["key"] = api_key

response = requests.get(url, params=params)

# Changing JSON response into DataFrame
data = response.json()
df_census = pd.DataFrame(data[1:], columns=data[0])

# Rename columns to human-readable names
df_census = df_census.rename(columns={
    "B01003_001E": "total_population",
    "B19013_001E": "median_income"
})

# Create 11-digit GEOID to match shapefiles later
df_census["GEOID"] = df_census["state"] + df_census["county"] + df_census["tract"]

# Save to local folder
os.makedirs("data/census", exist_ok=True)
df_census.to_csv("data/census/mecklenburg_census.csv", index=False)

df_census.head()

,NAME,total_population,median_income,state,county,tract,GEOID
0,Census Tract 1.01; Mecklenburg County; North C...,1148,101587,37,119,000101,37119000101
1,Census Tract 1.02; Mecklenburg County; North C...,2741,123650,37,119,000102,37119000102
2,Census Tract 1.03; Mecklenburg County; North C...,2042,131398,37,119,000103,37119000103
3,Census Tract 1.04; Mecklenburg County; North C...,1619,109896,37,119,000104,37119000104
4,Census Tract 3.01; Mecklenburg County; North C...,954,82500,37,119,000301,37119000301


In [ ]:
# Load afternoon heat traverse points
gdf_heat = gpd.read_file(r"C:\Users\Fiorella\OneDrive\VSCODE PROJECTS\Projects\DTSC 2\Personal Project Part 1\data\raw_data\heat_island\charlotte-north-carolina_af_trav.shp")
gdf_heat.head()

,t_f,rh,hi_f,geometry
0,94.6,40.0,98.3,POINT (-80.80071 35.19492)
1,93.8,41.1,97.4,POINT (-80.79847 35.19024)
2,94.6,39.7,98.1,POINT (-80.80764 35.17184)
3,96.7,29.0,96.5,POINT (-80.82773 35.20721)
4,97.0,30.4,97.5,POINT (-80.82879 35.20487)


: 